In [40]:
import pandas as pd

In [41]:
from pathlib import Path

print(Path.cwd())

c:\Users\wiame\Desktop\digital-twin-churn\data\ml_data


In [67]:
df = pd.read_csv("../raw/telco_churn.csv")
df.shape
df.dtypes
df.head()
df.tail()
df.info()
df.describe()
df.nunique()
"""df.isna().sum()
df.duplicated().sum()"""

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

'df.isna().sum()\ndf.duplicated().sum()'

In [68]:
df.describe()
df.nunique()
df.columns
df.index
(df.isna().sum() / len(df)) * 100
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

In [69]:
## Fix total charges with blank values for custommers who have tenure = 0 and convert Totalcharges from str to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"],errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0) # # tenure=0 clients → 0 charged so far, not missing data


In [70]:
## Drop the identifier because its not usefull in the training
customer_ids = df["customerID"]   
df = df.drop(columns=["customerID"])

In [71]:
## Encode the target 
df["Churn"] = df["Churn"].map({"Yes" : 1,"No" : 0})
df.nunique()

gender                 2
SeniorCitizen          2
Partner                2
Dependents             2
tenure                73
PhoneService           2
MultipleLines          3
InternetService        3
OnlineSecurity         3
OnlineBackup           3
DeviceProtection       3
TechSupport            3
StreamingTV            3
StreamingMovies        3
Contract               3
PaperlessBilling       2
PaymentMethod          4
MonthlyCharges      1585
TotalCharges        6531
Churn                  2
dtype: int64

In [72]:
## Encode binary categorical columns
binary_columns = ["Partner","Dependents","PhoneService","PaperlessBilling"] # plus the gender column


for col in binary_columns : 
    df[col] = df[col].map({"Yes":1,"No":0})

## Encoding the gender column 

df["gender"] = df["gender"].map({"Male" : 1,"Female" : 0})

In [73]:
## Handling the non binary categorical columns (service columns) using One Hot Encoding

service_columns = ["MultipleLines","InternetService","OnlineSecurity","OnlineBackup","DeviceProtection","TechSupport","StreamingTV","StreamingMovies","Contract","PaymentMethod"]

df = pd.get_dummies(df,columns = service_columns,drop_first = True) ## we drop the first column in the values of a column to evit redundancy

In [74]:
## Feature Engineering

df["avg_monthly_spend"] = df["TotalCharges"] / df["tenure"].replace(0,1)

df["n_services"] = df[[c for c in df.columns if c.startswith(("OnlineSecurity_", "OnlineBackup_",
                        "DeviceProtection_", "TechSupport_", "StreamingTV_", "StreamingMovies_"))
                        and c.endswith("Yes")]].sum(axis=1)

df["tenure_bucket"] = pd.cut(df["tenure"], bins=[0, 12, 24, 48, 72], labels=["0-1y", "1-2y", "2-4y", "4-6y"])

## Encoding the column tenure_bucket
df = pd.get_dummies(df, columns=["tenure_bucket"], drop_first=True)

In [75]:
## Checking output class imbalance

df["Churn"].value_counts(normalize = True)

Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64

## 73.5% of customers did not churn and 26.5% of customers churned

In [76]:
## scale positive weight 
scale_pos_weight = (df["Churn"] == 0).sum() / (df["Churn"] == 1).sum()   
scale_pos_weight

np.float64(2.7683253076511503)

In [77]:
## Splitting the data
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split( 
    X, y, test_size = 0.2, random_state= 42
)

In [79]:
 ## saving the processed table
processed = X.copy()
processed["Churn"] = y
processed["customerID"] = customer_ids.values


processed.to_csv("../processed/telco_clean.csv",index= False)